---
## 1.1 Ladda in data

In [1]:
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Data/raw_marketing_campaign.csv', sep='\t')
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


---
## 1.2 Första översikt

In [2]:
# Hur många rader och kolumner har datan?
print(df.shape)

(2240, 29)


In [3]:
# Hur ser de första raderna ut?
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [4]:
# Vilka datatyper har kolumnerna?
print(df.dtypes)

ID                       int64
Year_Birth               int64
Education                  str
Marital_Status             str
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer                str
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Z_CostContact            int64
Z_Revenue                int64
Response                 int64
dtype: object


In [5]:
# Visar kolumner med eventuellt saknade värden
df.isnull().sum()[df.isnull().sum() > 0]

Income    24
dtype: int64

---
## 1.3 Hantera saknade värden

In [6]:
# Tar bort rader där Income saknas
df = df.dropna(subset=['Income'])

# Verifierar att det det fungerade
print(df.shape)

(2216, 29)


---
## 1.4 Hantera felaktiga format och outliers

In [7]:
# Ersätt skräpvärden och dubbletter i Marital_Status
df["Marital_Status"] = df["Marital_Status"].replace({
    "Alone": "Single",
    "Absurd": "Single",
    "YOLO": "Single"
})

# Verifiera att det ser rätt ut
print(df["Marital_Status"].value_counts())

Marital_Status
Married     857
Together    573
Single      478
Divorced    232
Widow        76
Name: count, dtype: int64


In [8]:
# Kollar ifall det finns extrema outliers i Income. Tar bort värden som är mer än 3 standardavvikelser från medelvärdet.
mean = df["Income"].mean()
std = df["Income"].std()
upper_limit = mean + 3 * std

print(f"Medelvärde: {mean:.0f}")
print(f"Standardavvikelse: {std:.0f}")
print(f"Övre gräns (mean + 3*std): {upper_limit:.0f}")
print(f"Antal kunder över gränsen: {(df['Income'] > upper_limit).sum()}")

Medelvärde: 52247
Standardavvikelse: 25173
Övre gräns (mean + 3*std): 127766
Antal kunder över gränsen: 8


In [9]:
# Tar bort de 8 kunderna med extrema inkomster över 127 766
df = df[df["Income"] <= upper_limit]

# Verifiera
print(df.shape)
print(f"Max Income efter städning: {df['Income'].max():.0f}")

(2208, 29)
Max Income efter städning: 113734


In [10]:
# Tar bort kolumner som inte tillför något analytiskt värde:
# - Z_CostContact och Z_Revenue har samma värde för alla kunder (konstanter)
# - ID är bara ett löpnummer och beskriver inte kunden som person
df = df.drop(columns=["Z_CostContact", "Z_Revenue", "ID"], errors="ignore")

# Verifiera att kolumnerna är borttagna — ska nu visa 26 kolumner
print(df.shape)

(2208, 26)


## 1.5 Skapar nya variabler

In [11]:
# Vi skapar nya variabler utifrån befintliga kolumner för att göra datan mer användbar i vår analys.
# Skapar ålder utifrån födelseår
df["Age"] = datetime.now().year - df["Year_Birth"]

# Verifiera att det ser rimligt ut
print(df["Age"].describe())

count    2208.000000
mean       57.192935
std        11.991913
min        30.000000
25%        49.000000
50%        56.000000
75%        67.000000
max       133.000000
Name: Age, dtype: float64


In [12]:
# Tar bort kunder med orimlig ålder (över 95 år)
df = df[df["Age"] <= 95]

# Verifiera
print(df.shape)
print(f"Max ålder efter städning: {df['Age'].max():.0f}")

(2205, 27)
Max ålder efter städning: 86


In [13]:
# Översikt över min och max för alla numeriska kolumner
df.describe().loc[["min", "max"]].T

,min,max
Year_Birth,1940.0,1996.0
Income,1730.0,113734.0
Kidhome,0.0,2.0
Teenhome,0.0,2.0
Recency,0.0,99.0
MntWines,0.0,1493.0
MntFruits,0.0,199.0
MntMeatProducts,0.0,1725.0
MntFishProducts,0.0,259.0
MntSweetProducts,0.0,262.0


---
## 1.6 Spara rent dataset

In [14]:
# Sparar det rengjorda datasetet för användning i nästa notebook
df.to_csv("Data/clean_customer_data.csv", index=False)

print(f"Rent dataset sparat!")
print(f"Shape: {df.shape}")

Rent dataset sparat!
Shape: (2205, 27)
